# Feature Engineering for Better Predictions

**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**

## Learning Objectives

- Create interaction features to capture combined effects
- Generate polynomial features for non-linear relationships
- Encode categorical variables effectively
- Build lagged features from temporal data
- Create domain-specific soccer features
- Understand feature selection techniques
- Avoid common feature engineering pitfalls


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression, RFE

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


## Why Feature Engineering Matters

**Quote:** "Coming up with features is difficult, time-consuming, requires expert knowledge. 'Applied machine learning' is basically feature engineering." - Andrew Ng

**In soccer analytics:**
- Raw stats (shots, passes) are just the starting point
- **Interaction effects** matter (e.g., shots × accuracy)
- **Context** is crucial (home vs away, opponent strength)
- **Temporal patterns** reveal form and momentum
- **Domain knowledge** creates the best features


## Load Base Data

In [ ]:
# Generate realistic match data
np.random.seed(42)
n_matches = 300

teams = ['Arsenal', 'Chelsea', 'Liverpool', 'Man City', 'Man United', 'Tottenham', 
         'Leicester', 'West Ham', 'Everton', 'Wolves']

df = pd.DataFrame({
    'match_id': range(n_matches),
    'team': np.random.choice(teams, n_matches),
    'opponent': np.random.choice(teams, n_matches),
    'home': np.random.choice([0, 1], n_matches),
    'shots': np.random.randint(5, 20, n_matches),
    'shots_on_target': np.random.randint(2, 12, n_matches),
    'possession': np.random.uniform(35, 70, n_matches),
    'pass_accuracy': np.random.uniform(70, 92, n_matches),
    'xg': np.random.uniform(0.5, 3.5, n_matches),
    'opponent_xg': np.random.uniform(0.5, 3.5, n_matches),
    'match_week': np.random.randint(1, 39, n_matches)
})

# Generate realistic goals
df['goals'] = (
    df['xg'] * 0.8 + 
    df['shots_on_target'] * 0.05 + 
    df['home'] * 0.3 +
    np.random.normal(0, 0.5, n_matches)
).clip(0, 6).round().astype(int)

print("Base Dataset:")
print(df.head())
print(f"\
Shape: {df.shape}")
print(f"Features: {df.columns.tolist()}")


## 1. Interaction Features

**Concept:** Capture combined effects of two or more features.

**Soccer examples:**
- `shots × accuracy` = Quality of shooting
- `possession × pass_accuracy` = Effective possession
- `home × opponent_strength` = Home advantage against strong teams


In [ ]:
# Create interaction features
df['shot_quality'] = df['shots_on_target'] / (df['shots'] + 1)  # Avoid division by zero
df['effective_possession'] = df['possession'] * df['pass_accuracy'] / 100
df['xg_difference'] = df['xg'] - df['opponent_xg']
df['shot_efficiency'] = df['xg'] / (df['shots'] + 1)
df['attacking_threat'] = df['shots_on_target'] * df['xg']

print("Interaction Features Created:")
print(df[['shot_quality', 'effective_possession', 'xg_difference', 'shot_efficiency', 'attacking_threat']].head())

# Visualize impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['shots_on_target'], df['goals'], alpha=0.3, label='Shots on Target')
axes[0].set_xlabel('Shots on Target')
axes[0].set_ylabel('Goals')
axes[0].set_title('Raw Feature')
axes[0].legend()

axes[1].scatter(df['attacking_threat'], df['goals'], alpha=0.3, color='green', label='Attacking Threat')
axes[1].set_xlabel('Attacking Threat (SoT × xG)')
axes[1].set_ylabel('Goals')
axes[1].set_title('Interaction Feature')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\
Interaction features often show stronger correlations with the target.")


## 2. Polynomial Features

**Concept:** Capture non-linear relationships by adding squared, cubed terms.

**Example:** `xG²` might capture that very high xG matches are even more likely to produce goals.


In [ ]:
# Create polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
features_for_poly = ['xg', 'shots_on_target']
X_poly = poly.fit_transform(df[features_for_poly])

# Get feature names
poly_feature_names = poly.get_feature_names_out(features_for_poly)
df_poly = pd.DataFrame(X_poly, columns=poly_feature_names)

print("Polynomial Features:")
print(df_poly.head())
print(f"\
Original features: {len(features_for_poly)}")
print(f"Polynomial features: {len(poly_feature_names)}")
print(f"\
New features include: {list(poly_feature_names)}")

# Compare models
X_original = df[features_for_poly]
y = df['goals']

X_train_orig, X_test_orig, y_train, y_test = train_test_split(X_original, y, test_size=0.25, random_state=42)
X_train_poly, X_test_poly, _, _ = train_test_split(df_poly, y, test_size=0.25, random_state=42)

# Linear model with original features
model_orig = LinearRegression()
model_orig.fit(X_train_orig, y_train)
r2_orig = model_orig.score(X_test_orig, y_test)

# Linear model with polynomial features
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
r2_poly = model_poly.score(X_test_poly, y_test)

print(f"\
Model Performance:")
print(f"  Original features R²: {r2_orig:.3f}")
print(f"  Polynomial features R²: {r2_poly:.3f}")
print(f"  Improvement: {(r2_poly - r2_orig)*100:.1f}%")


## 3. Categorical Feature Encoding

**Challenge:** Machine learning models need numbers, not text.

**Methods:**
- **Label Encoding:** Assign numbers (0, 1, 2...) - implies order
- **One-Hot Encoding:** Create binary columns for each category
- **Target Encoding:** Replace with mean target value
- **Frequency Encoding:** Replace with category frequency


In [ ]:
# 1. Target Encoding (team strength based on average goals)
team_strength = df.groupby('team')['goals'].mean().to_dict()
opponent_strength = df.groupby('opponent')['goals'].mean().to_dict()

df['team_strength'] = df['team'].map(team_strength)
df['opponent_strength'] = df['opponent'].map(opponent_strength)

print("Target Encoding - Team Strength:")
for team, strength in sorted(team_strength.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {team}: {strength:.2f} goals/match")

# 2. One-Hot Encoding for home/away
df['is_home'] = df['home']
df['is_away'] = 1 - df['home']

print(f"\
One-Hot Encoding created: is_home, is_away")

# 3. Frequency Encoding
team_freq = df['team'].value_counts(normalize=True).to_dict()
df['team_frequency'] = df['team'].map(team_freq)

print(f"\
Frequency Encoding: Teams by match frequency")
print(df.groupby('team')['team_frequency'].first().sort_values(ascending=False).head())


## 4. Lagged Features (Temporal)

**Concept:** Use past performance to predict future outcomes.

**Soccer examples:**
- Last 3 matches average goals
- Form (rolling average)
- Momentum (trend)
- Head-to-head history


In [ ]:
# Sort by team and match week
df = df.sort_values(['team', 'match_week']).reset_index(drop=True)

# Create lagged features
df['goals_last_match'] = df.groupby('team')['goals'].shift(1)
df['goals_last_3_avg'] = df.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
)
df['goals_last_5_avg'] = df.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

# Form indicator (recent performance)
df['recent_form'] = df['goals_last_3_avg'] - df['goals_last_5_avg']

print("Lagged Features:")
print(df[['team', 'match_week', 'goals', 'goals_last_match', 'goals_last_3_avg', 'recent_form']].head(10))

# Visualize form
team_example = 'Arsenal'
team_data = df[df['team'] == team_example].head(15)

plt.figure(figsize=(12, 6))
plt.plot(team_data['match_week'], team_data['goals'], 'o-', label='Actual Goals', linewidth=2)
plt.plot(team_data['match_week'], team_data['goals_last_3_avg'], 's--', label='Last 3 Avg', linewidth=2)
plt.xlabel('Match Week')
plt.ylabel('Goals')
plt.title(f'{team_example} - Goals and Form')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\
Lagged features capture team form and momentum.")


## 5. Domain-Specific Features

**Soccer-specific features** based on tactical knowledge:


In [ ]:
# 1. Defensive solidity
df['defensive_solidity'] = 1 / (df['opponent_xg'] + 0.1)  # Lower opponent xG = better defense

# 2. Attacking efficiency
df['attacking_efficiency'] = df['xg'] / (df['shots'] + 1)

# 3. Possession quality
df['possession_quality'] = (df['possession'] * df['pass_accuracy']) / 100

# 4. Match importance (later in season = more important)
df['match_importance'] = df['match_week'] / 38  # Normalize to 0-1

# 5. Strength differential
df['strength_diff'] = df['team_strength'] - df['opponent_strength']

# 6. Expected goal difference
df['expected_goal_diff'] = df['xg'] - df['opponent_xg']

print("Domain-Specific Features:")
print(df[['defensive_solidity', 'attacking_efficiency', 'possession_quality', 
          'strength_diff', 'expected_goal_diff']].head())

# Correlation with goals
feature_cols = ['defensive_solidity', 'attacking_efficiency', 'possession_quality', 
                'strength_diff', 'expected_goal_diff']
correlations = df[feature_cols + ['goals']].corr()['goals'].drop('goals').sort_values(ascending=False)

print(f"\
Correlation with Goals:")
for feat, corr in correlations.items():
    print(f"  {feat}: {corr:.3f}")


## 6. Feature Selection

**Problem:** Too many features can cause overfitting.

**Methods:**
1. **Filter methods:** Select based on statistical tests
2. **Wrapper methods:** Select based on model performance
3. **Embedded methods:** Feature selection during training


In [ ]:
# Prepare feature set (drop NaN from lagged features)
df_clean = df.dropna()

feature_candidates = [
    'shots', 'shots_on_target', 'possession', 'pass_accuracy', 'xg', 'opponent_xg',
    'home', 'shot_quality', 'effective_possession', 'xg_difference', 'shot_efficiency',
    'attacking_threat', 'team_strength', 'opponent_strength', 'goals_last_3_avg',
    'recent_form', 'defensive_solidity', 'attacking_efficiency', 'strength_diff'
]

X_full = df_clean[feature_candidates]
y_full = df_clean['goals']

print(f"Total candidate features: {len(feature_candidates)}")

# 1. Filter Method: SelectKBest
selector_filter = SelectKBest(score_func=f_regression, k=10)
X_filtered = selector_filter.fit_transform(X_full, y_full)
selected_features_filter = [feature_candidates[i] for i in selector_filter.get_support(indices=True)]

print(f"\
Filter Method (SelectKBest) - Top 10 features:")
for i, feat in enumerate(selected_features_filter, 1):
    print(f"  {i}. {feat}")

# 2. Wrapper Method: Recursive Feature Elimination
model_rfe = LinearRegression()
selector_rfe = RFE(model_rfe, n_features_to_select=10, step=1)
selector_rfe.fit(X_full, y_full)
selected_features_rfe = [feature_candidates[i] for i in range(len(feature_candidates)) if selector_rfe.support_[i]]

print(f"\
Wrapper Method (RFE) - Top 10 features:")
for i, feat in enumerate(selected_features_rfe, 1):
    print(f"  {i}. {feat}")


## Compare: All Features vs. Selected Features

In [ ]:
# Split data
X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y_full, test_size=0.25, random_state=42)
X_train_sel, X_test_sel, _, _ = train_test_split(X_full[selected_features_filter], y_full, test_size=0.25, random_state=42)

# Model with all features
model_all = LinearRegression()
model_all.fit(X_train_full, y_train)
r2_all = model_all.score(X_test_full, y_test)
rmse_all = np.sqrt(mean_squared_error(y_test, model_all.predict(X_test_full)))

# Model with selected features
model_sel = LinearRegression()
model_sel.fit(X_train_sel, y_train)
r2_sel = model_sel.score(X_test_sel, y_test)
rmse_sel = np.sqrt(mean_squared_error(y_test, model_sel.predict(X_test_sel)))

print("Model Comparison:")
print(f"\
All Features ({len(feature_candidates)}):")
print(f"  R²: {r2_all:.3f}")
print(f"  RMSE: {rmse_all:.3f}")
print(f"\
Selected Features ({len(selected_features_filter)}):")
print(f"  R²: {r2_sel:.3f}")
print(f"  RMSE: {rmse_sel:.3f}")

if r2_sel >= r2_all * 0.98:  # Within 2%
    print(f"\
✅ Feature selection maintained performance with {len(feature_candidates) - len(selected_features_filter)} fewer features!")
else:
    print(f"\
⚠️ Feature selection reduced performance. Consider keeping more features.")


## Common Pitfalls to Avoid

### 1. Data Leakage
**Problem:** Using information that wouldn't be available at prediction time.

**Example:** Including "goals in next match" as a feature 😱

**Solution:** Only use features available before the match.


In [ ]:
# WRONG: Using future information
# df['goals_next_match'] = df.groupby('team')['goals'].shift(-1)  # DON'T DO THIS!

# RIGHT: Using only past information
df['goals_previous_match'] = df.groupby('team')['goals'].shift(1)  # OK

print("✅ Always ensure lagged features use .shift(positive_number)")
print("❌ Never use .shift(negative_number) - that's future data!")


### 2. Multicollinearity
**Problem:** Highly correlated features can make models unstable.

**Solution:** Remove or combine correlated features.


In [ ]:
# Check correlation matrix
corr_matrix = X_full.corr()

# Find highly correlated pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

if high_corr_pairs:
    print("⚠️ Highly Correlated Feature Pairs (|r| > 0.8):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
    print("\
Consider removing one feature from each pair.")
else:
    print("✅ No severe multicollinearity detected.")


### 3. Overfitting with Too Many Features
**Problem:** Model memorizes training data instead of learning patterns.

**Solution:** Use cross-validation and regularization.


In [ ]:
# Compare training vs test performance
from sklearn.model_selection import cross_val_score

model = LinearRegression()

# Cross-validation scores
cv_scores = cross_val_score(model, X_full, y_full, cv=5, scoring='r2')

print("Cross-Validation Performance:")
print(f"  Mean R²: {cv_scores.mean():.3f}")
print(f"  Std R²: {cv_scores.std():.3f}")
print(f"\
If std is high (>0.1), model may be overfitting.")

if cv_scores.std() > 0.1:
    print("⚠️ High variance detected. Consider:")
    print("  - Reducing number of features")
    print("  - Using regularization (Ridge/Lasso)")
    print("  - Collecting more data")
else:
    print("✅ Model shows stable performance across folds.")


## Summary

In this notebook, we:

1. ✅ Created interaction features for combined effects
2. ✅ Generated polynomial features for non-linearity
3. ✅ Encoded categorical variables multiple ways
4. ✅ Built lagged features from temporal data
5. ✅ Designed domain-specific soccer features
6. ✅ Applied feature selection techniques
7. ✅ Learned to avoid common pitfalls

## Key Takeaways

- **Feature engineering > Model selection** in most cases
- **Interaction features** capture combined effects
- **Polynomial features** handle non-linear relationships
- **Lagged features** capture temporal patterns and form
- **Domain knowledge** creates the most powerful features
- **Feature selection** prevents overfitting
- **Avoid data leakage** at all costs
- **Check for multicollinearity** in your features

## Feature Engineering Checklist

✅ Create interaction terms for related features
✅ Add polynomial terms if relationships are non-linear
✅ Encode categorical variables appropriately
✅ Include lagged features for temporal data
✅ Design domain-specific features using soccer knowledge
✅ Check for and remove highly correlated features
✅ Use feature selection to reduce dimensionality
✅ Validate no data leakage exists
✅ Test on holdout data to ensure generalization


## Practice Exercises

1. **Create More Interactions**: Try 3-way interactions (e.g., home × xG × opponent_strength)
2. **Time-Based Features**: Add day of week, month, season period
3. **Rolling Statistics**: Create rolling min/max, not just averages
4. **Opponent-Specific Features**: Head-to-head history, tactical matchups
5. **Feature Importance**: Use Random Forest to rank feature importance
6. **Automated Feature Engineering**: Research and try automated tools like Featuretools
7. **Real Data Application**: Apply these techniques to actual match data
